# **Running CASREL with Demo Dataset**

This notebook demonstrates a preliminary evaluation of CASREL using a small demonstration dataset. The primary objectives are:

1. Assess the technical feasibility and computational efficiency of CASREL
2. Provide a template for implementation and code execution
3. Demonstrate the workflow with a simplified dataset

**Important Notes**:

*   This demonstration uses a highly reduced dataset, consisting of only dozens of randomly sampled cells and a limited number of AS events from the complete dataset.
*   The demonstration dataset serves purely as a proof-of-concept example.
*   You can run this notebook in **Google Colab** (recommended) or on your **local machine** (macOS / Windows / Linux).

# **Environment Setup**

## Prerequisites

- **Python**: 3.8 – 3.10 (3.10 recommended)
- **Git**: installed and available in PATH
- **OS**: macOS, Windows, or Linux

### Recommended: Create a virtual environment before running this notebook

```bash
# Using conda (recommended)
conda create -n casrel python=3.10 -y
conda activate casrel
pip install jupyter ipykernel
python -m ipykernel install --user --name casrel --display-name "CASREL (Python 3.10)"

# Or using venv
python3.10 -m venv casrel_env
source casrel_env/bin/activate  # macOS/Linux
# casrel_env\Scripts\activate   # Windows
pip install jupyter ipykernel
python -m ipykernel install --user --name casrel --display-name "CASREL (Python 3.10)"
```

Then select the `CASREL (Python 3.10)` kernel in VS Code or Jupyter.

In [ ]:
import sys
import platform
import subprocess
import os

print("="*60)
print("ENVIRONMENT INFORMATION")
print("="*60)
print(f"Python version : {sys.version}")
print(f"Platform       : {platform.system()} {platform.release()}")
print(f"Architecture   : {platform.machine()}")
print(f"Working dir    : {os.getcwd()}")
print("="*60)

# Check Python version
py_ver = sys.version_info
if py_ver < (3, 8) or py_ver >= (3, 11):
    print("\n⚠️  WARNING: Python 3.8–3.10 is recommended. "
          f"You are using Python {py_ver.major}.{py_ver.minor}.{py_ver.micro}")
    print("   Some dependencies may not be compatible.")
else:
    print(f"\n✅ Python {py_ver.major}.{py_ver.minor}.{py_ver.micro} — compatible.")

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("\n📍 Running in Google Colab.")
else:
    print(f"\n📍 Running locally on {platform.system()}.")

## Clone the CASREL repository

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/xryanglab/CASREL.git"
REPO_DIR = "CASREL"

if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL} ...")
    result = subprocess.run(["git", "clone", REPO_URL], 
                           capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ Git clone failed:\n{result.stderr}")
        print("\nPlease ensure git is installed:")
        print("  macOS  : brew install git  (or install Xcode Command Line Tools)")
        print("  Windows: https://git-scm.com/download/win")
        print("  Linux  : sudo apt-get install git")
        raise RuntimeError("git clone failed")
    else:
        print("✅ Repository cloned successfully.")
else:
    print(f"✅ Directory '{REPO_DIR}' already exists, skipping clone.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Install dependencies

In [ ]:
import subprocess
import sys

# Install pinned dependencies to ensure reproducibility
DEPENDENCIES = [
    "pandas==1.4.2",
    "numpy==1.25.2",
    "tqdm==4.66.1",
    "seaborn==0.13.2",
    "matplotlib==3.5.3",
    "scikit-learn==1.2.2",
    "lightgbm==4.1.0",
    "shap==0.43.0",
]

print("Installing dependencies...\n")
cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + DEPENDENCIES
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("⚠️  Some packages may have failed to install.")
    print(result.stderr)
    print("\nTrying with relaxed version constraints...")
    # Fallback: install without strict version pinning
    DEPENDENCIES_RELAXED = [
        "pandas>=1.4,<2.0",
        "numpy>=1.23,<1.26",
        "tqdm",
        "seaborn",
        "matplotlib>=3.5",
        "scikit-learn>=1.2,<1.4",
        "lightgbm>=4.0,<4.5",
        "shap>=0.42,<0.45",
    ]
    cmd2 = [sys.executable, "-m", "pip", "install", "--quiet"] + DEPENDENCIES_RELAXED
    result2 = subprocess.run(cmd2, capture_output=True, text=True)
    if result2.returncode != 0:
        print(f"❌ Installation failed:\n{result2.stderr}")
        raise RuntimeError("Dependency installation failed.")

print("✅ All dependencies installed successfully.")

In [ ]:
# Verify installed package versions
import pandas as pd
import numpy as np
import sklearn
import lightgbm
import shap
import matplotlib
import seaborn as sns

print("Installed package versions:")
print(f"  pandas       : {pd.__version__}")
print(f"  numpy        : {np.__version__}")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  lightgbm     : {lightgbm.__version__}")
print(f"  shap         : {shap.__version__}")
print(f"  matplotlib   : {matplotlib.__version__}")
print(f"  seaborn      : {sns.__version__}")
print("\n✅ All packages imported successfully.")

# **Data Preprocessing**

In [ ]:
import subprocess
import sys

print("Running data preprocessing...")
result = subprocess.run(
    [sys.executable, "src/preprocess.py",
     "--splice_file", "demo/demo_filter",
     "--gene_file", "demo/demo_rbp_exp.csv",
     "--output_dir", "demo/"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"❌ Preprocessing failed:\n{result.stderr}")
else:
    print("✅ Preprocessing completed successfully.")

# **Train the Models and Explain the Models**

This step trains LightGBM classifiers for each AS site across 5-fold cross-validation and computes SHAP values. This may take several minutes depending on your hardware.

In [ ]:
import subprocess
import sys

print("Training models (this may take 10-30 minutes)...")
print("Progress will be shown below:\n")

process = subprocess.Popen(
    [sys.executable, "src/train.py", "-k", "-1", "-i", "demo/", "-o", "demo/output/"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

# Stream output line by line
fold_count = 0
site_count = 0
for line in process.stdout:
    line = line.strip()
    if line.startswith("Start processing:"):
        fold_count += 1
        site_count = 0
        print(f"\n{'='*50}")
        print(f"  {line} (Fold {fold_count}/5)")
        print(f"{'='*50}")
    elif line.startswith("Complete processing:"):
        print(f"  ✅ {line} — {site_count} sites processed")
    elif "Trained site index:" in line:
        site_count += 1
        # Print progress every 50 sites to avoid flooding output
        idx = int(line.split(":")[-1].strip())
        if (idx + 1) % 50 == 0 or idx == 0:
            print(f"    ... processed site {idx + 1}")

process.wait()
if process.returncode == 0:
    print(f"\n✅ Training completed successfully for all 5 folds.")
else:
    print(f"\n❌ Training failed with return code {process.returncode}")

# **Screening of AS Regulatory Factors**

In [ ]:
import subprocess
import sys

print("Running regulatory factor screening...")
result = subprocess.run(
    [sys.executable, "src/reg_summary.py", "-b", "demo/output/"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"❌ Screening failed:\n{result.stderr}")
else:
    print("✅ Regulatory factor screening completed.")

# **View Results**

In [ ]:
import pandas as pd
import os

output_file = "demo/output/SHAP_summary.csv"

if os.path.isfile(output_file):
    df = pd.read_csv(output_file)
    print(f"Results saved to: {output_file}")
    print(f"Total regulatory relationships found: {len(df)}")
    print(f"\nFirst 10 rows:")
    display(df.head(10)) if 'display' in dir() else print(df.head(10).to_string())
else:
    print(f"❌ Output file not found at: {output_file}")
    print("Please check that the previous steps completed without errors.")

# **Outputs**

The intermediate and result files are saved in the `demo/output/` folder. The primary output of CASREL is a CSV file named `SHAP_summary.csv`, containing the predicted RNA splicing regulatory circuitry.

#### File Structure

| Column | Description |
|--------|------------|
| RBP | RNA binding protein (regulatory factor) |
| AS_Event | Alternative splicing event (splicing site) regulated by the RBP |
| Contribution | Predicted regulatory contribution score |
| Regulation | Regulatory direction (`+` for promotion, `-` for inhibition) |

This output file provides a comprehensive overview of the predicted regulatory relationships between RBPs and their target splicing events, including both the strength and direction of regulation.

---

# **Troubleshooting**

| Issue | Solution |
|-------|----------|
| `git: command not found` | Install Git: macOS (`xcode-select --install`), Windows (https://git-scm.com), Linux (`sudo apt install git`) |
| `ModuleNotFoundError` | Re-run the dependency installation cell; ensure you're using the correct kernel |
| Version conflicts | Create a fresh virtual environment as described in the setup section |
| `MemoryError` during training | Close other applications; the demo dataset requires ~2 GB RAM |
| Notebook kernel crashes | Restart kernel and re-run from the beginning |
| Permission errors on Windows | Run VS Code / terminal as administrator, or use a path without spaces |

**For Google Colab users**: Simply click "Runtime → Run all" — no additional setup is needed. The notebook auto-detects the Colab environment.